# Understanding the TVSD neural DataLoader

This tutorial follows monkey-F TVSD activity from the raw MATLAB file to the tensors returned by the experiment DataLoaders. It answers four practical questions:

1. How are the 22,248 unique training images and 100 repeated test images represented in `ALLMAT`?
2. Does one cached neural trace exactly match baseline correction and temporal binning performed directly from `ALLMUA`?
3. How is each neural presentation paired with the correct cached I-JEPA representation?
4. Which statistics are fitted on training data before the targets reach the model?

The normal pipeline reads the compact baseline-corrected NumPy cache. The 55 GB MATLAB file is opened only for one selected presentation/channel audit; the complete `ALLMUA` array is never loaded into memory.

```text
Raw ALLMUA [source time, presentations, raw channels]
    |
    +-- official headstage permutation and IT channel selection
    +-- presentation-specific -100 to 0 ms baseline subtraction
    +-- non-overlapping temporal means
    v
Target cache [presentations, cached time bins, physical IT channels]
    |
    +-- ALLMAT train_idx/test_idx feature lookup
    +-- seeded fit/validation split of unique training presentations
    +-- fit-only channel standardization
    +-- optional fit-only feature centering and robust neuron min-max
    v
DataLoader batch:
    I-JEPA features [batch, layers, embedding]
    neural targets  [batch, time, neurons]
```


## Axis ledger

| Stage | Shape | Meaning |
|---|---|---|
| HDF5 `ALLMUA` | `[source_time, presentations, raw_channels]` | 1 kHz MUA before ROI selection |
| cached targets | `[presentations, cached_time, physical_channels]` | baseline-corrected and binned responses |
| cached stimulus features | `[unique_images, layers, embedding]` | one representation per train or test image |
| materialized subset features | `[presentations, layers, embedding]` | repeated test images are expanded through `test_idx` |
| DataLoader target batch | `[batch, time, neurons]` | standardized and optionally preprocessed model targets |

TVSD's official training pool contains one presentation per image. Its official test pool contains 30 presentations of each of 100 images. The test pool is not used for fitting or checkpoint selection.


In [ ]:
import sys
from dataclasses import dataclass, field
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate the repository whether Jupyter starts in the project root or scripts folder.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate config.yaml.")
# end if the project root is unavailable

project_src_path = str((PROJECT_ROOT / "python_scripts" / "src").resolve())
while project_src_path in sys.path:
    sys.path.remove(project_src_path)
# end while the project source is already registered
sys.path.insert(0, project_src_path)

from IT_recap.tvsd import (  # noqa: E402
    TVSD_METADATA_COLUMNS,
    get_tvsd_monkey_f_area_channels,
    load_tvsd_metadata,
    make_tvsd_headstage_mapping,
)
from IT_recap.tvsd_experiments import (  # noqa: E402
    gather_presentation_features,
    load_cached_data,
    load_project_paths,
    make_tensor_loader,
    prepare_timebin_data,
    resolve_cache_paths,
    resolve_device,
    select_window_bin_indices,
    standardize_targets,
)
from project_specific_utils.dataloader import (  # noqa: E402
    apply_neural_preprocessing,
)


In [ ]:
@dataclass
class Cfg:
    # Prepared TVSD caches.
    env: str | None = None
    mua_file_name: str = "f_THINGS_MUA_trials.mat"
    feature_archive_name: str = (
        "tvsd_monkeyF_ijepa_vith14_1k_224_features.npz"
    )

    # Cached neural target and tutorial window.
    area: str = "IT"
    target_fs: int = 100
    time_start_ms: float = 0.0
    time_end_ms: float = 200.0
    window_start_ms: float = 0.0
    window_end_ms: float = 200.0
    timebin_ms: float = 10.0

    # Optional target preprocessing fitted on training presentations only.
    center_neural_features: bool = True
    robust_minmax_neurons: bool = False
    robust_percentile_range: tuple[float, float] = (1.0, 99.0)
    clip_robust_minmax: bool = True

    # Cached I-JEPA depths selected from the feature archive.
    model_name: str = "ijepa_vith14_1k"
    layer_names: list[str] = field(default_factory=lambda: [
        "encoder.layer.4.output.dense",
        "encoder.layer.17.output.dense",
        "encoder.layer.27.output.dense",
    ])

    # Seeded split and DataLoader behavior.
    validation_fraction: float = 0.1
    random_seed: int = 0
    batch_size: int = 8
    num_workers: int = 0
    smoke_test: bool = False

    # One raw-to-cache audit and one repeated-test visualization.
    example_test_image_id: int = 1  # one-based ALLMAT test_idx
    example_repetition_index: int = 0
    example_neuron_index: int = 0  # local physical channel within area
    n_repetitions_to_plot: int = 5

    # Required by the shared preparation and reporting path.
    noise_ceiling_resamples: int = 40
    device: str = "auto"
# EOC


cfg = Cfg()
cfg


## 1. Resolve and inspect the caches

The target-cache filename records the area, cached response window, and sampling rate. `np.load(..., mmap_mode="r")` keeps the 616 MB neural cache on disk and pages in only requested slices. The feature archive contains both stimulus spaces: unique training images and the 100 identities used in the repeated test pool.


In [ ]:
paths = load_project_paths(cfg)
device = resolve_device(cfg.device)
cache_paths = resolve_cache_paths(cfg, paths)

for cache_name, cache_path in cache_paths.items():
    if not cache_path.is_file():
        raise FileNotFoundError(f"Missing {cache_name} cache: {cache_path}")
    # end if a required cache is unavailable
# end for configured cache

targets, train_features, test_features, allmat = load_cached_data(cfg, paths)
_, source_time_ms = load_tvsd_metadata(cache_paths["mua"])

if targets.shape != (len(allmat), 20, 320):
    raise RuntimeError(f"Unexpected IT target shape: {targets.shape}.")
# end if the prepared target cache is unexpected
if train_features.shape[1:] != test_features.shape[1:]:
    raise ValueError("Train and test feature dimensions disagree.")
# end if the two stimulus feature spaces differ

print(f"raw MATLAB file: {cache_paths['mua']}")
print(f"target cache [presentations, time, IT channels]: {targets.shape}")
print(f"target storage: {type(targets).__name__}, dtype={targets.dtype}")
print(f"ALLMAT [presentations, metadata columns]: {allmat.shape}")
print(f"source time base: {source_time_ms[0]:g} to {source_time_ms[-1]:g} ms")
print(f"selected train features: {train_features.shape}")
print(f"selected test features: {test_features.shape}")


## 2. Decode the presentation metadata

Each `ALLMAT` row describes one neural presentation:

```text
trial_idx, train_idx, test_idx, rep, count, day
```

MATLAB stimulus identifiers are one-based. Exactly one of `train_idx` or `test_idx` is nonzero. A training row indexes `train_features[train_idx - 1]`; a test row indexes `test_features[test_idx - 1]`.


In [ ]:
metadata = {
    column_name: allmat[:, column_index]
    for column_index, column_name in enumerate(TVSD_METADATA_COLUMNS)
}
official_train_trial_indices = np.flatnonzero(metadata["train_idx"] > 0)
official_test_trial_indices = np.flatnonzero(metadata["test_idx"] > 0)

train_image_ids = metadata["train_idx"][official_train_trial_indices]
test_image_ids = metadata["test_idx"][official_test_trial_indices]
unique_test_ids, test_repetition_counts = np.unique(
    test_image_ids,
    return_counts=True,
)

np.testing.assert_array_equal(
    np.sort(train_image_ids),
    np.arange(1, len(train_features) + 1),
)
np.testing.assert_array_equal(
    unique_test_ids,
    np.arange(1, len(test_features) + 1),
)
if not np.all(test_repetition_counts == 30):
    raise ValueError("Every official test image should have 30 presentations.")
# end if the repeated-test protocol is incomplete

print(f"official training presentations: {len(official_train_trial_indices):,}")
print(f"unique training image IDs: {len(np.unique(train_image_ids)):,}")
print(f"official test presentations: {len(official_test_trial_indices):,}")
print(f"unique test image IDs: {len(unique_test_ids)}")
print(
    f"test repetitions per image: {test_repetition_counts.min()}-"
    f"{test_repetition_counts.max()}"
)


## 3. Audit one cached target against raw `ALLMUA`

The prepared cache was created by:

1. mapping the selected physical IT channel back to its raw HDF5 channel;
2. subtracting that presentation's mean response from −100 to 0 ms;
3. averaging consecutive 1 kHz samples into `target_fs` bins.

The following cell reproduces those operations for one test presentation and one IT channel. It reads only one 300-sample trace from the 55 GB file.


In [ ]:
matching_test_trials = np.flatnonzero(
    metadata["test_idx"] == cfg.example_test_image_id
)
if not 0 <= cfg.example_repetition_index < len(matching_test_trials):
    raise IndexError("example_repetition_index is unavailable for this test image.")
# end if the requested repetition is invalid
example_trial_index = int(
    matching_test_trials[cfg.example_repetition_index]
)

physical_channels, array_numbers = get_tvsd_monkey_f_area_channels(cfg.area)
if not 0 <= cfg.example_neuron_index < len(physical_channels):
    raise IndexError("example_neuron_index is outside the selected area.")
# end if the requested local channel is invalid
physical_channel = int(physical_channels[cfg.example_neuron_index])
raw_channel = int(make_tvsd_headstage_mapping()[physical_channel])

with h5py.File(cache_paths["mua"], "r") as h5file:
    raw_trace = np.asarray(
        h5file["ALLMUA"][:, example_trial_index, raw_channel],
        dtype=np.float32,
    )
# end with raw MUA file

baseline_mask = (source_time_ms >= -100.0) & (source_time_ms < 0.0)
response_mask = (
    (source_time_ms >= cfg.time_start_ms)
    & (source_time_ms < cfg.time_end_ms)
)
baseline_mean = raw_trace[baseline_mask].mean(dtype=np.float32)
response_trace = raw_trace[response_mask]

source_bin_width = 1000 // cfg.target_fs
if len(response_trace) % source_bin_width != 0:
    raise ValueError("The raw response does not form complete cached bins.")
# end if the cached binning would leave a partial bin
manual_cached_response = response_trace.reshape(
    -1,
    source_bin_width,
).mean(axis=1, dtype=np.float32) - baseline_mean
cached_response = np.asarray(
    targets[example_trial_index, :, cfg.example_neuron_index],
    dtype=np.float32,
)
np.testing.assert_allclose(
    cached_response,
    manual_cached_response,
    rtol=1e-5,
    atol=1e-5,
)

cached_bin_centers_ms = cfg.time_start_ms + (
    np.arange(len(cached_response)) + 0.5
) * 1000.0 / cfg.target_fs

figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(source_time_ms, raw_trace, color="0.55")
axes[0].axvspan(-100, 0, color="tab:blue", alpha=0.12, label="baseline")
axes[0].axvspan(
    cfg.time_start_ms,
    cfg.time_end_ms,
    color="tab:orange",
    alpha=0.12,
    label="cached response",
)
axes[0].set_xlabel("Time from stimulus onset (ms)")
axes[0].set_ylabel("Raw MUA")
axes[0].set_title(
    f"Trial {example_trial_index}, raw channel {raw_channel}"
)
axes[0].legend(frameon=False)

axes[1].plot(
    cached_bin_centers_ms,
    manual_cached_response,
    color="black",
    linewidth=3,
    label="manual raw-file calculation",
)
axes[1].plot(
    cached_bin_centers_ms,
    cached_response,
    color="tab:orange",
    linestyle="--",
    label="saved target cache",
)
axes[1].set_xlabel("Cached time-bin center (ms)")
axes[1].set_ylabel("Baseline-corrected MUA")
axes[1].set_title(
    f"{cfg.area} neuron {cfg.example_neuron_index}; traces overlap"
)
axes[1].legend(frameon=False)
figure.tight_layout()
plt.show()

print(f"physical {cfg.area} arrays: {array_numbers}")
print(f"physical channel: {physical_channel}; raw HDF5 channel: {raw_channel}")
print(f"source samples per cached bin: {source_bin_width}")
print("PASS: the target cache exactly reproduces the raw-file calculation.")


## 4. Prove the stimulus-feature lookup

Unlike the natural-image raster loader, TVSD does not use image filenames to align presentations. The nonzero `train_idx` or `test_idx` in each `ALLMAT` row directly selects the corresponding one-based stimulus representation.


In [ ]:
example_metadata = allmat[example_trial_index]
example_feature = gather_presentation_features(
    train_features,
    test_features,
    allmat,
    np.asarray([example_trial_index]),
)[0]
example_test_idx = int(example_metadata[2])
expected_feature = test_features[example_test_idx - 1]

np.testing.assert_array_equal(example_feature, expected_feature)
print(
    "example ALLMAT row: "
    + ", ".join(
        f"{name}={int(value)}"
        for name, value in zip(TVSD_METADATA_COLUMNS, example_metadata)
    )
)
print(f"paired feature [layers, embedding]: {example_feature.shape}")
print("PASS: test_idx selects the exact cached test-image representation.")


## 5. Select the model window and prepare all subsets

The target cache uses 10 ms bins at 100 Hz. `window_start_ms` and `window_end_ms` select cached bins, and `timebin_ms` can average consecutive cached bins into a coarser model target.

`prepare_timebin_data` then performs the real experiment pipeline:

- keeps the official repeated-test pool untouched;
- splits the 22,248 unique training presentations into fit and validation subsets;
- estimates channel mean and standard deviation from fit presentations only;
- optionally centers every `(neuron, time-bin)` feature from the fit subset;
- optionally applies robust per-neuron min-max scaling using fit percentiles;
- materializes aligned feature/target arrays and DataLoaders.


In [ ]:
bin_indices, covered_ms = select_window_bin_indices(
    targets.shape[1],
    cfg.target_fs,
    cfg.time_start_ms,
    cfg.window_start_ms,
    cfg.window_end_ms,
)
cached_bin_ms = 1000.0 / cfg.target_fs
group_size_float = cfg.timebin_ms / cached_bin_ms
group_size = int(round(group_size_float))
if not np.isclose(group_size_float, group_size):
    raise ValueError("timebin_ms must be a multiple of the cached bin width.")
# end if the requested output bins cannot be formed
if len(bin_indices) % group_size != 0:
    raise ValueError("The selected window leaves a partial output bin.")
# end if the model window cannot form complete output bins

data = prepare_timebin_data(
    cfg,
    targets,
    train_features,
    test_features,
    allmat,
    device,
)

print(f"selected cached bins: {bin_indices.tolist()}")
print(f"covered model window: {covered_ms[0]:g}-{covered_ms[1]:g} ms")
print(f"cached bins per model bin: {group_size}")
for split_name, split_indices in data["indices"].items():
    print(
        f"{split_name:>10}: {len(split_indices):,} presentations | "
        f"features {data['subset_features'][split_name].shape} | "
        f"targets {data['subset_targets'][split_name].shape}"
    )
# end for prepared subset


## 6. Reproduce one final model target

This audit starts from one cached training target, applies the selected window and re-binning, then applies the exact fit-only channel standardization and optional neural preprocessing stored by `prepare_timebin_data`. The result must equal the corresponding row in `subset_targets["train"]`.


In [ ]:
training_trial_index = int(data["indices"]["train"][0])
selected_cached_target = np.asarray(
    targets[training_trial_index, bin_indices, :],
    dtype=np.float32,
)
manual_timebin_target = selected_cached_target.reshape(
    len(bin_indices) // group_size,
    group_size,
    targets.shape[2],
).mean(axis=1)

channel_mean, channel_scale = data["channel_standardization"]
manual_model_target = standardize_targets(
    manual_timebin_target[None, :, :],
    channel_mean,
    channel_scale,
)[0]

neural_preprocessing_stats = data["neural_preprocessing_stats"]
if neural_preprocessing_stats is not None:
    manual_model_target = apply_neural_preprocessing(
        manual_model_target.T[:, :, None],
        neural_preprocessing_stats,
    )[:, :, 0].T
# end if optional neural preprocessing is enabled

prepared_model_target = data["subset_targets"]["train"][0]
np.testing.assert_allclose(
    prepared_model_target,
    manual_model_target,
    rtol=1e-6,
    atol=1e-6,
)

print(f"channel mean and scale: {channel_mean.shape}, {channel_scale.shape}")
print(
    "optional preprocessing: "
    f"center_features={cfg.center_neural_features}, "
    f"robust_minmax_neurons={cfg.robust_minmax_neurons}, "
    f"percentiles={cfg.robust_percentile_range}"
)
print(f"final target [time, neurons]: {prepared_model_target.shape}")
print("PASS: manual preprocessing equals the prepared model target.")


## 7. Audit the exact DataLoader batches

The training DataLoader normally shuffles presentations. For an exact audit, the cell below creates an otherwise identical unshuffled loader for each subset and verifies its first batch against the materialized arrays. Shuffling changes only visit order; it cannot change a feature-target pairing.


In [ ]:
audit_loaders = {}
for split_name in ("train", "validation", "test"):
    audit_loader = make_tensor_loader(
        data["subset_features"][split_name],
        data["subset_targets"][split_name],
        cfg,
        shuffle=False,
    )
    audit_loaders[split_name] = audit_loader
    feature_batch, target_batch = next(iter(audit_loader))

    expected_features = torch.as_tensor(
        data["subset_features"][split_name][: cfg.batch_size],
        dtype=torch.float32,
    )
    expected_targets = torch.as_tensor(
        data["subset_targets"][split_name][: cfg.batch_size],
        dtype=torch.float32,
    )
    torch.testing.assert_close(feature_batch, expected_features, rtol=0, atol=0)
    torch.testing.assert_close(target_batch, expected_targets, rtol=0, atol=0)

    print(
        f"{split_name:>10} batch: features {tuple(feature_batch.shape)}, "
        f"targets {tuple(target_batch.shape)}"
    )
# end for prepared subset

print("PASS: every unshuffled first batch matches its materialized sources.")


## 8. Inspect repeated test presentations

All presentations of one official test image receive the same cached I-JEPA representation, but retain separate neural targets. This is the crucial difference between the unique training pool and repeated test pool.


In [ ]:
prepared_test_image_ids = data["scoring"]["test_image_ids"]
example_test_positions = np.flatnonzero(
    prepared_test_image_ids == cfg.example_test_image_id - 1
)
if len(example_test_positions) != 30:
    raise ValueError("Expected exactly 30 prepared repetitions.")
# end if the selected test image has incomplete repetitions

expected_repeated_feature = test_features[cfg.example_test_image_id - 1]
np.testing.assert_array_equal(
    data["subset_features"]["test"][example_test_positions],
    np.broadcast_to(
        expected_repeated_feature,
        (
            len(example_test_positions),
            *expected_repeated_feature.shape,
        ),
    ),
)

repeated_targets = data["subset_targets"]["test"][
    example_test_positions
]  # [repetitions, time, neurons]
example_neuron_repetitions = repeated_targets[
    :, :, cfg.example_neuron_index
]
mean_test_response = example_neuron_repetitions.mean(axis=0)
model_bin_centers_ms = data["bin_edges_ms"][:-1] + cfg.timebin_ms / 2.0

n_repetitions_to_plot = min(
    cfg.n_repetitions_to_plot,
    len(example_neuron_repetitions),
)
plt.figure(figsize=(8, 4))
for repetition_index in range(n_repetitions_to_plot):
    plt.plot(
        model_bin_centers_ms,
        example_neuron_repetitions[repetition_index],
        color="0.7",
        alpha=0.8,
        label="single repetitions" if repetition_index == 0 else None,
    )
# end for displayed repetition
plt.plot(
    model_bin_centers_ms,
    mean_test_response,
    color="tab:blue",
    linewidth=3,
    label="mean of 30 repetitions",
)
plt.xlabel("Model time-bin center (ms)")
plt.ylabel("Final model-target units")
plt.title(
    f"Test image {cfg.example_test_image_id}, "
    f"{cfg.area} neuron {cfg.example_neuron_index}"
)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

print(
    f"one repeated feature: {expected_repeated_feature.shape}; "
    f"repeated targets: {repeated_targets.shape}"
)
print("PASS: repeated targets share one exact stimulus representation.")


## Take-home interpretation

- A TVSD sample is a neural presentation, not necessarily a unique stimulus.
- `train_idx` maps unique training presentations to `train_features[train_idx - 1]`.
- `test_idx` maps all 30 repetitions of a test image to the same `test_features[test_idx - 1]`.
- The saved neural cache is already ROI-selected, baseline-corrected, and temporally binned.
- The model window may select or further average cached time bins.
- Channel standardization is fitted only on the fit subset.
- Optional feature-wise centering and robust per-neuron min-max statistics are also fitted only on the fit subset.
- Validation and official-test targets are transformed with the saved fit statistics; they are never refitted.
- Training shuffling changes presentation order only. The feature-target pairing is materialized before batching.
- The official repeated-test pool remains untouched by the seeded fit/validation split.

The raw-to-cache assertion in Section 3 and the final-target assertion in Section 6 are the two strongest checks to retain when changing area, time window, temporal resolution, or neural preprocessing.
